# 02. Harmonic presence fields and pinch-out boundaries

This notebook plots the stored harmonic presence fields and pinch-out boundaries for the three middle geological units. Presence values are dimensionless interpolation results, not calibrated probabilities.

**Input:** `demo_model.npz`, generated by `run_model.py`. The input directory is selected by `HG3D_MODEL_DIR` (default: `outputs/model/`).

**Output:** `harmonic_fields_and_fronts.png`. Figures are displayed and saved to `HG3D_FIGURE_DIR` (default: `outputs/figures/`).

**Interpretation:** The fields and boundaries are read from the model package. This notebook does not solve the harmonic interpolation problem.


## 1. Imports and configuration


In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

# Project paths.
PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "run_model.py").is_file() and (path / "notebooks").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Start this notebook from the HarmonicGeo3D directory or a subdirectory."
    )

# Input data.
MODEL_DATA_DIR = Path(
    os.environ.get("HG3D_MODEL_DIR", PROJECT_ROOT / "outputs" / "model")
)

CM_TO_INCH = 1 / 2.54
FONT_FAMILY = ["Arial", "DejaVu Sans"]
FIGURE_WIDTH_CM, FIGURE_HEIGHT_CM = 15, 6.25
TICK_INTERVAL_M = 200.0
SELECTED_BODY_NAMES = ("body-01", "body-02", "body-03")
FONT_SIZES = {
    "label": 6.0,
    "tick": 4.0,
    "tag": 7,
}

# The colour scale maps presence values from dark to light grey-green.
PRESENCE_COLORS = ("#607E78", "#88AAA3", "#BCCFC7", "#E4E9DE")
PRESENCE_CMAP = LinearSegmentedColormap.from_list(
    "sage_presence", PRESENCE_COLORS, N=256
)
FRONT_COLOR = "#315C6B"  # Pinch-out boundary colour.

BOREHOLE_COLOR = "#1D2430"
PANEL_LABELS = ("(a)", "(b)", "(c)")
PLOT_RC = {
    "font.family": FONT_FAMILY,
    "axes.unicode_minus": False,
}

# Figure output.
FIGURE_DIR = Path(
    os.environ.get("HG3D_FIGURE_DIR", PROJECT_ROOT / "outputs" / "figures")
)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

## 2. Data loading and unit selection


In [ ]:
with np.load(MODEL_DATA_DIR / "demo_model.npz", allow_pickle=False) as model_data:
    initial_nodes_xy = model_data["initial_nodes_xy"]
    initial_triangles = model_data["initial_triangles"]
    body_names = model_data["body_names"].astype(str)
    unit_presence_values = model_data["unit_presence_values"]
    front_points_xy = model_data["front_points_xy"]
    front_segments = model_data["front_segments"]
    front_point_offsets = model_data["front_point_offsets"]
    front_segment_offsets = model_data["front_segment_offsets"]

# Stable unit names determine the panel order.
body_name_to_index = {name: index for index, name in enumerate(body_names)}
missing_bodies = [
    name for name in SELECTED_BODY_NAMES if name not in body_name_to_index
]
if missing_bodies:
    raise ValueError(
        f"The model is missing the selected geological units: {missing_bodies}"
    )
selected_body_indices = [body_name_to_index[name] for name in SELECTED_BODY_NAMES]

## 3. Plot harmonic presence fields and pinch-out boundaries


In [ ]:
with plt.rc_context(PLOT_RC):
    fig, axes = plt.subplots(
        1,
        3,
        figsize=(FIGURE_WIDTH_CM * CM_TO_INCH, FIGURE_HEIGHT_CM * CM_TO_INCH),
        dpi=600,
        squeeze=False,
        gridspec_kw={"wspace": 0.01},
        layout="constrained",
    )
    axes = axes.ravel()

    # Small numerical errors can place dimensionless presence values outside [0, 1].
    selected_presence_values = np.clip(
        unit_presence_values[:, selected_body_indices], 0.0, 1.0
    )
    fill_levels = np.linspace(0.0, 1.0, 9)
    # Extended colour levels include triangles with values constant at zero or one.
    # This display adjustment leaves the stored fields and geometry unchanged.
    fill_levels[0] -= 1e-12
    fill_levels[-1] += 1e-12

    x_padding = np.ptp(initial_nodes_xy[:, 0]) * 0.03
    y_padding = np.ptp(initial_nodes_xy[:, 1]) * 0.03
    x_limits = (
        initial_nodes_xy[:, 0].min() - x_padding,
        initial_nodes_xy[:, 0].max() + x_padding,
    )
    y_limits = (
        initial_nodes_xy[:, 1].min() - y_padding,
        initial_nodes_xy[:, 1].max() + y_padding,
    )

    for panel, (body_index, ax) in enumerate(zip(selected_body_indices, axes)):
        image = ax.tricontourf(
            initial_nodes_xy[:, 0],
            initial_nodes_xy[:, 1],
            initial_triangles,
            selected_presence_values[:, panel],
            levels=fill_levels,
            cmap=PRESENCE_CMAP,
            vmin=0.0,
            vmax=1.0,
            zorder=1,
            alpha=0.9,
        )
        ax.scatter(
            initial_nodes_xy[:, 0],
            initial_nodes_xy[:, 1],
            s=4,
            facecolor=BOREHOLE_COLOR,
            edgecolor="white",
            linewidth=0.35,
            zorder=4,
        )

        points = front_points_xy[
            front_point_offsets[body_index] : front_point_offsets[body_index + 1]
        ]
        segments = front_segments[
            front_segment_offsets[body_index] : front_segment_offsets[body_index + 1]
        ]
        for segment in segments:
            ax.plot(
                points[segment, 0],
                points[segment, 1],
                color=FRONT_COLOR,
                linewidth=0.6,
                zorder=5,
                linestyle="dashed",
            )

        for spine in ax.spines.values():
            spine.set_linewidth(0.6)
            spine.set_color("#333333")

        ax.set_xlabel("X (m)", family=FONT_FAMILY, fontsize=FONT_SIZES["label"])
        ax.set_ylabel(
            "Y (m)" if panel == 0 else "",
            family=FONT_FAMILY,
            fontsize=FONT_SIZES["label"],
        )
        ax.set_axisbelow(True)
        ax.tick_params(
            axis="x",
            which="both",
            top=True,
            bottom=True,
            labeltop=False,
            labelbottom=True,
            direction="in",
            labelsize=FONT_SIZES["tick"],
            length=2,
            width=0.5,
        )
        ax.tick_params(
            axis="y",
            which="both",
            left=True,
            right=True,
            labelright=False,
            labelleft=panel == 0,
            direction="in",
            labelsize=FONT_SIZES["tick"],
            length=2,
            width=0.5,
            rotation=90,
        )
        ax.ticklabel_format(style="plain", useOffset=False)

        ax.grid(axis="both", alpha=0.8, color="lightgray", linewidth=0.25)

        # Panel letters identify the selected units.
        ax.text(
            0.02,
            0.98,
            PANEL_LABELS[panel],
            transform=ax.transAxes,
            family=FONT_FAMILY,
            fontsize=FONT_SIZES["tag"],
            ha="left",
            va="top",
            zorder=5,
        )

        # Equal XY scales preserve the geometry of the map.
        ax.set_xlim(*x_limits)
        ax.set_ylim(*y_limits)
        ax.set_aspect("equal", adjustable="box")
        ax.set_box_aspect((y_limits[1] - y_limits[0]) / (x_limits[1] - x_limits[0]))

    colorbar = fig.colorbar(
        image,
        ax=axes,
        pad=0.01,
        fraction=0.035,
        shrink=0.8,
    )
    colorbar.set_ticks(np.linspace(0.0, 1.0, 5))
    colorbar.ax.tick_params(
        direction="in", labelsize=FONT_SIZES["tick"], length=2, width=0.5
    )

    fig.savefig(FIGURE_DIR / "harmonic_fields_and_fronts.png", dpi=600)
    plt.show()